# Accuracy measures

ISO/IEC 25024 `Acc-I-*` and ISO/IEC 5259-2 `Acc-ML-*` measures: whether a value is correct — inside a
required range, inside the column's domain, or plausible given the rest of the record.

In [1]:
import polars as pl

from dqmeasure import (
    DataAccuracyRange,
    RiskOfDataSetInaccuracy,
    SemanticDataAccuracy,
    SyntacticDataAccuracy,
)

## DataAccuracyRange
**"Are data values inside the required interval?"**

Column-scoped. `fit` learns the interval `[min, max]` from the clean data.

In [2]:
clean = pl.DataFrame({"temperature": [18.0, 21.5, 19.0, 22.0, 20.5]})
dirty = pl.DataFrame({"temperature": [20.0, 41.0, 19.5, -3.0, 21.0]})

measure = DataAccuracyRange("temperature").fit(clean)
measure.low_, measure.high_

(18.0, 22.0)

In [3]:
measure.predict(dirty)

temperature
f64
1.0
0.0
1.0
0.0
1.0


In [4]:
measure.score(dirty)

0.6

Two of five readings fall outside the 18-to-22 °C range learned from clean data.

## RiskOfDataSetInaccuracy
**"Which values are outliers, at risk of being inaccurate?"**

Uses a robust z-score (median and MAD, scaled to sigma units) learned from clean data. Lower is better: it counts risk, not correctness.

In [5]:
readings = pl.DataFrame({"latency_ms": [12, 14, 11, 13, 15, 12, 14, 13, 250, -80]})
clean, dirty = readings[:8], readings  # first 8 rows are the clean reference

measure = RiskOfDataSetInaccuracy("latency_ms").fit(clean)
measure.center_, measure.scale_

(13.0, 1.4826)

In [6]:
measure.predict(dirty)

latency_ms
f64
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
1.0


In [7]:
measure.score(dirty)

0.2

Two of ten readings sit far enough from the median to count as outliers.

## SyntacticDataAccuracy
**"Is the value a member of the column's valid domain?"**

`fit` learns the domain as the distinct values seen in clean data — no notion of range or type, just membership.

In [8]:
status = pl.DataFrame({"status": ["open", "closed", "pending", "open", "closed"]})
new_status = pl.DataFrame({"status": ["open", "closed", "archived", "pending", "cancelled"]})

measure = SyntacticDataAccuracy("status").fit(status)
measure.domain_

{'closed', 'open', 'pending'}

In [9]:
measure.predict(new_status)

status
f64
1.0
1.0
0.0
1.0
0.0


In [10]:
measure.score(new_status)

0.6

`archived` and `cancelled` never appeared in the clean data, so two of five values fail.

## SemanticDataAccuracy
**"Is the value plausible, given the rest of the record and real-world knowledge?"**

A language model judges each value in context, few-shot-prompted with clean example rows. Needs a running OpenAI-compatible endpoint — the default targets a local Ollama server (`ollama serve`, with `llama3.2:3b` pulled). The cells below are not executed in this notebook; run them yourself once Ollama is up.

In [ ]:
cities = pl.DataFrame({"city": ["Paris", "Tokyo", "Cairo", "Oslo"], "country": ["France", "Japan", "Egypt", "Norway"]})
mismatched = pl.DataFrame(
    {"city": ["Paris", "Tokyo", "Cairo", "Berlin"], "country": ["France", "Japan", "Egypt", "Brazil"]}
)  # Berlin is not in Brazil

measure = SemanticDataAccuracy("country").fit(cities)
measure.predict(mismatched)

In [ ]:
measure.score(mismatched)

Expect the model to flag `Brazil` for Berlin as implausible: `score` around `0.75`.